# Notebook 18 — Ratings Semantics and Availability

## Bounded question

> What do the runner-level `or`, `rpr` and `ts` fields represent in the source, how consistently are they populated across time, jurisdictions and race types, and which values can be compared or derived safely without inventing equivalence between official, private and performance-rating scales?

## Initial governed scope

This notebook investigates three runner-level source fields:

- `or`
- `rpr`
- `ts`

The source-field governance register assigns all three to the `performance_market_and_value` family and requires their raw values to be preserved.

The investigation begins with source profiling only. At this stage, no assumption is made that:

- `or` always represents an official rating under every jurisdiction or authority;
- `rpr` and `ts` use the same scale or methodology;
- blank values, dashes and zeros have equivalent meanings;
- ratings are comparable across jurisdictions, racing codes or periods;
- a missing value in one field can be derived from either of the others;
- any rating represents an eligibility condition;
- the three values were produced contemporaneously or from the same information set.

Raw values, parsed numeric values and availability states will remain separate. Any later interpretation must preserve source-field identity, jurisdiction and period context, unresolved cases and physical lineage.

## Stage 1 — Source lineage and governed population

This stage establishes the immutable source, read-only controls and expected population before profiling `or`, `rpr` or `ts`.

The source is:

- database: `data/raw/form_2015-present/form_2015-present/raceform.db`
- table: `data`
- governed row predicate: `rowid <> 1`

The established source population is:

- 1,851,285 governed runner rows;
- 189,043 provisional races;
- 37 source columns;
- provisional race identity: `date + course + off`.

The first code cell will open SQLite in read-only mode, confirm the source schema and reconcile the governed runner and provisional-race counts. It will not yet interpret or profile the three rating fields.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd


PROJECT_ROOT = Path.cwd().resolve().parent
SOURCE_DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"
RACE_KEY_COLUMNS = ["date", "course", "off"]
RATING_FIELDS = ["or", "rpr", "ts"]

EXPECTED_RUNNER_ROWS = 1_851_285
EXPECTED_PROVISIONAL_RACES = 189_043
EXPECTED_SOURCE_COLUMNS = 37

if not SOURCE_DB_PATH.exists():
    raise FileNotFoundError(f"Source database not found: {SOURCE_DB_PATH}")

connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

source_columns = pd.read_sql_query(
    f"PRAGMA table_info({SOURCE_TABLE})",
    connection,
)

population = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS runner_rows,
        COUNT(
            DISTINCT
            CAST(date AS TEXT) || '|' ||
            CAST(course AS TEXT) || '|' ||
            CAST(off AS TEXT)
        ) AS provisional_races
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

observed_runner_rows = int(population.loc[0, "runner_rows"])
observed_provisional_races = int(population.loc[0, "provisional_races"])
observed_source_columns = len(source_columns)

available_columns = set(source_columns["name"])
missing_rating_fields = sorted(set(RATING_FIELDS) - available_columns)

assert observed_runner_rows == EXPECTED_RUNNER_ROWS
assert observed_provisional_races == EXPECTED_PROVISIONAL_RACES
assert observed_source_columns == EXPECTED_SOURCE_COLUMNS
assert not missing_rating_fields, (
    f"Missing expected rating fields: {missing_rating_fields}"
)

print("Governed source population confirmed")

display(
    pd.DataFrame(
        {
            "measure": [
                "source database",
                "source table",
                "data-row predicate",
                "runner rows",
                "provisional races",
                "source columns",
                "rating fields present",
            ],
            "value": [
                str(SOURCE_DB_PATH.relative_to(PROJECT_ROOT)),
                SOURCE_TABLE,
                DATA_ROW_PREDICATE,
                observed_runner_rows,
                observed_provisional_races,
                observed_source_columns,
                ", ".join(RATING_FIELDS),
            ],
        }
    )
)

Governed source population confirmed


,measure,value
0,source database,data/raw/form_2015-present/form_2015-present/r...
1,source table,data
2,data-row predicate,rowid <> 1
3,runner rows,1851285
4,provisional races,189043
5,source columns,37
6,rating fields present,"or, rpr, ts"


In [2]:
# Locate the permanent source-field governance register.
FIELD_GOVERNANCE_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "source_field_governance.csv"
)

if not FIELD_GOVERNANCE_PATH.exists():
    raise FileNotFoundError(
        f"Field-governance register not found: {FIELD_GOVERNANCE_PATH}"
    )

# Load the existing governed field inventory without altering it.
field_governance = pd.read_csv(FIELD_GOVERNANCE_PATH)

# Define the columns required to inspect the inherited governance position.
required_governance_columns = {
    "source_field",
    "declared_type",
    "grain",
    "field_family",
    "raw_preservation",
    "blank_policy",
    "dash_policy",
    "zero_policy",
    "governed_by",
    "status",
}

# Fail explicitly if the permanent register no longer has the expected schema.
missing_governance_columns = sorted(
    required_governance_columns - set(field_governance.columns)
)

if missing_governance_columns:
    raise ValueError(
        "Field-governance register is missing required columns: "
        f"{missing_governance_columns}"
    )

# Retain only the three rating fields and the governance attributes relevant
# to this investigation. Reindexing preserves the intended field order.
rating_governance = (
    field_governance.loc[
        field_governance["source_field"].isin(RATING_FIELDS),
        [
            "source_field",
            "declared_type",
            "grain",
            "field_family",
            "raw_preservation",
            "blank_policy",
            "dash_policy",
            "zero_policy",
            "governed_by",
            "status",
        ],
    ]
    .copy()
    .set_index("source_field")
    .reindex(RATING_FIELDS)
    .reset_index()
)

# Confirm that all three expected governance rows are present and remain
# governed as runner-level, raw-preserved fields awaiting semantic study.
assert len(rating_governance) == len(RATING_FIELDS)
assert rating_governance["source_field"].tolist() == RATING_FIELDS
assert rating_governance["grain"].eq("runner").all()
assert rating_governance["raw_preservation"].eq("required").all()
assert rating_governance["field_family"].eq(
    "performance_market_and_value"
).all()
assert rating_governance["status"].eq("pending_semantics").all()

display(rating_governance)

,source_field,declared_type,grain,field_family,raw_preservation,blank_policy,dash_policy,zero_policy,governed_by,status
0,or,INTEGER,runner,performance_market_and_value,required,field_not_supplied,unavailable_rating,possible_sentinel,Notebook 10,pending_semantics
1,rpr,INTEGER,runner,performance_market_and_value,required,field_not_supplied,unavailable_rating,possible_sentinel,Notebook 10,pending_semantics
2,ts,INTEGER,runner,performance_market_and_value,required,field_not_supplied,unavailable_rating,possible_sentinel,Notebook 10,pending_semantics


## Stage 2 — Confirm governed field ownership and provisional policies

Before examining the values, this stage checks the existing source-field governance rows for `or`, `rpr` and `ts`.

The purpose is to confirm:

- runner-level grain;
- declared SQLite type;
- field-family ownership;
- raw-value preservation requirements;
- current blank, dash and zero policies;
- the semantic status inherited from Notebook 10.

These governance rows are starting constraints rather than final conclusions.

In particular:

- `unavailable_rating` is only the existing provisional interpretation of a dash;
- `possible_sentinel` does not establish that zero is missing;
- the shared field family does not establish that the three ratings share a scale;
- the declared SQLite type does not prove that every stored value is numeric.

Notebook 18 may refine these policies only after the raw source behaviour has been profiled.

In [3]:
# Locate the permanent source-field governance register.
FIELD_GOVERNANCE_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "source_field_governance.csv"
)

# Stop immediately if the governed reference file is missing.
if not FIELD_GOVERNANCE_PATH.exists():
    raise FileNotFoundError(
        f"Field-governance register not found: {FIELD_GOVERNANCE_PATH}"
    )

# Load the existing field-governance decisions without modifying the file.
field_governance = pd.read_csv(FIELD_GOVERNANCE_PATH)

# These columns contain the inherited constraints and provisional policies
# needed before inspecting the raw values in or, rpr and ts.
required_governance_columns = {
    "source_field",
    "declared_type",
    "grain",
    "field_family",
    "raw_preservation",
    "blank_policy",
    "dash_policy",
    "zero_policy",
    "governed_by",
    "status",
}

# Detect schema drift explicitly rather than allowing later selections or
# assertions to fail with a less informative error.
missing_governance_columns = sorted(
    required_governance_columns - set(field_governance.columns)
)

if missing_governance_columns:
    raise ValueError(
        "Field-governance register is missing required columns: "
        f"{missing_governance_columns}"
    )

# Select only the three rating fields and the governance attributes relevant
# to this study. Reindexing preserves the intended or, rpr, ts display order.
rating_governance = (
    field_governance.loc[
        field_governance["source_field"].isin(RATING_FIELDS),
        [
            "source_field",
            "declared_type",
            "grain",
            "field_family",
            "raw_preservation",
            "blank_policy",
            "dash_policy",
            "zero_policy",
            "governed_by",
            "status",
        ],
    ]
    .copy()
    .set_index("source_field")
    .reindex(RATING_FIELDS)
    .reset_index()
)

# Confirm that every expected rating field has exactly one governance row.
assert len(rating_governance) == len(RATING_FIELDS)
assert rating_governance["source_field"].tolist() == RATING_FIELDS

# Protect the inherited design constraints before any semantic interpretation:
# all three fields must remain runner-level, raw-preserved and assigned to the
# performance/market/value family.
assert rating_governance["grain"].eq("runner").all()
assert rating_governance["raw_preservation"].eq("required").all()
assert rating_governance["field_family"].eq(
    "performance_market_and_value"
).all()

# Confirm that Notebook 18 is beginning from unresolved semantic status rather
# than accidentally treating an earlier provisional policy as a final rule.
assert rating_governance["status"].eq("pending_semantics").all()

# Display the inherited governance position for inspection before profiling
# any raw rating values.
display(rating_governance)

,source_field,declared_type,grain,field_family,raw_preservation,blank_policy,dash_policy,zero_policy,governed_by,status
0,or,INTEGER,runner,performance_market_and_value,required,field_not_supplied,unavailable_rating,possible_sentinel,Notebook 10,pending_semantics
1,rpr,INTEGER,runner,performance_market_and_value,required,field_not_supplied,unavailable_rating,possible_sentinel,Notebook 10,pending_semantics
2,ts,INTEGER,runner,performance_market_and_value,required,field_not_supplied,unavailable_rating,possible_sentinel,Notebook 10,pending_semantics


## Stage 3 — Profile raw storage and availability states

This stage examines the physical behaviour of `or`, `rpr` and `ts` before attempting to interpret any rating scale.

For each field, it will establish:

- SQLite storage classes;
- null values;
- blank text values;
- dash values;
- zero values;
- other populated values;
- distinct raw-value counts;
- numeric and non-numeric populated values;
- minimum and maximum safely numeric values.

The categories will be kept separate.

In particular:

- SQL `NULL` will not be combined automatically with blank text;
- a dash will not yet be assumed to mean the same thing as a blank;
- zero will remain visible rather than being converted to missing;
- declared `INTEGER` type will not be treated as proof that every stored value is numeric;
- numeric ranges will be calculated only from values that SQLite stores as integers or real numbers, or text values that pass an explicit integer-format check.

This first profile is field-level and source-wide. Temporal, jurisdiction, race-type and cross-field coverage will be investigated only after the raw vocabularies are understood.

In [4]:
# Build one source-wide profiling query for each rating field.
#
# SQLite permits values with different storage classes inside a column even
# when that column was declared as INTEGER. We therefore inspect typeof(value)
# rather than trusting the declared schema alone.
#
# The source field `or` is also a SQLite keyword, so every field identifier
# must be quoted before it is inserted into SQL.
rating_profile_queries = []

for field in RATING_FIELDS:
    # Quote the identifier separately from the literal field label.
    # Example: or becomes "or" when used as a column reference.
    quoted_field = f'"{field}"'

    rating_profile_queries.append(
        f"""
        SELECT
            '{field}' AS source_field,

            -- Count every governed runner row so all later categories can be
            -- reconciled back to the complete source population.
            COUNT(*) AS runner_rows,

            -- Preserve SQL NULL as its own physical missing-value state.
            SUM(
                CASE WHEN {quoted_field} IS NULL THEN 1 ELSE 0 END
            ) AS null_rows,

            -- Count empty or whitespace-only text separately from SQL NULL.
            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'text'
                     AND TRIM(CAST({quoted_field} AS TEXT)) = ''
                    THEN 1
                    ELSE 0
                END
            ) AS blank_text_rows,

            -- Count the exact dash vocabulary provisionally labelled as an
            -- unavailable rating in the governance register.
            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'text'
                     AND TRIM(CAST({quoted_field} AS TEXT)) = '-'
                    THEN 1
                    ELSE 0
                END
            ) AS dash_rows,

            -- Keep zero visible regardless of whether SQLite stored it as a
            -- number or as integer-like text.
            SUM(
                CASE
                    WHEN (
                        typeof({quoted_field}) IN ('integer', 'real')
                        AND CAST({quoted_field} AS REAL) = 0
                    )
                    OR (
                        typeof({quoted_field}) = 'text'
                        AND TRIM(CAST({quoted_field} AS TEXT))
                            IN ('0', '+0', '-0')
                    )
                    THEN 1
                    ELSE 0
                END
            ) AS zero_rows,

            -- Count values that are neither NULL, blank nor the exact dash.
            -- Zero remains included because it is physically populated.
            SUM(
                CASE
                    WHEN {quoted_field} IS NOT NULL
                     AND NOT (
                        typeof({quoted_field}) = 'text'
                        AND TRIM(CAST({quoted_field} AS TEXT)) IN ('', '-')
                     )
                    THEN 1
                    ELSE 0
                END
            ) AS populated_rows,

            -- Count distinct source representations without normalising them.
            COUNT(DISTINCT {quoted_field}) AS distinct_nonnull_raw_values,

            -- Record SQLite's physical storage classes for comparison.
            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'integer'
                    THEN 1
                    ELSE 0
                END
            ) AS integer_storage_rows,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'real'
                    THEN 1
                    ELSE 0
                END
            ) AS real_storage_rows,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'text'
                    THEN 1
                    ELSE 0
                END
            ) AS text_storage_rows,

            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'blob'
                    THEN 1
                    ELSE 0
                END
            ) AS blob_storage_rows,

            -- Count physically numeric values. Text is not included here,
            -- even where it appears numeric, so storage and parsing remain
            -- separate questions.
            SUM(
                CASE
                    WHEN typeof({quoted_field}) IN ('integer', 'real')
                    THEN 1
                    ELSE 0
                END
            ) AS physically_numeric_rows,

            -- Count populated text that is not blank or a dash. These values
            -- require vocabulary inspection before any parsing rule is made.
            SUM(
                CASE
                    WHEN typeof({quoted_field}) = 'text'
                     AND TRIM(CAST({quoted_field} AS TEXT)) NOT IN ('', '-')
                    THEN 1
                    ELSE 0
                END
            ) AS other_populated_text_rows,

            -- Calculate ranges only from physically numeric storage classes.
            -- This avoids silently coercing malformed text to zero.
            MIN(
                CASE
                    WHEN typeof({quoted_field}) IN ('integer', 'real')
                    THEN CAST({quoted_field} AS REAL)
                END
            ) AS min_physically_numeric_value,

            MAX(
                CASE
                    WHEN typeof({quoted_field}) IN ('integer', 'real')
                    THEN CAST({quoted_field} AS REAL)
                END
            ) AS max_physically_numeric_value

        FROM "{SOURCE_TABLE}"
        WHERE {DATA_ROW_PREDICATE}
        """
    )

# UNION ALL produces one comparable profile row for each source field while
# preserving the intended or, rpr, ts order.
rating_profile_sql = "\nUNION ALL\n".join(rating_profile_queries)

# Execute the completed source-wide query against the existing read-only
# connection.
rating_storage_profile = pd.read_sql_query(
    rating_profile_sql,
    connection,
)

# Confirm that the query returned exactly the three expected fields.
assert rating_storage_profile["source_field"].tolist() == RATING_FIELDS

# Every profile row must reconcile to the governed runner population.
assert rating_storage_profile["runner_rows"].eq(
    EXPECTED_RUNNER_ROWS
).all()

# SQLite storage classes must partition the complete population. SQL NULL is
# reported separately because typeof(NULL) returns 'null'.
for _, row in rating_storage_profile.iterrows():
    storage_partition = (
        int(row["null_rows"])
        + int(row["integer_storage_rows"])
        + int(row["real_storage_rows"])
        + int(row["text_storage_rows"])
        + int(row["blob_storage_rows"])
    )

    assert storage_partition == EXPECTED_RUNNER_ROWS, (
        f"Storage classes do not reconcile for {row['source_field']}: "
        f"{storage_partition:,} versus {EXPECTED_RUNNER_ROWS:,}"
    )

# Display the complete field-level profile before introducing any semantic or
# parsing interpretation.
display(rating_storage_profile)

,source_field,runner_rows,null_rows,blank_text_rows,dash_rows,zero_rows,populated_rows,distinct_nonnull_raw_values,integer_storage_rows,real_storage_rows,text_storage_rows,blob_storage_rows,physically_numeric_rows,other_populated_text_rows,min_physically_numeric_value,max_physically_numeric_value
0,or,1851285,0,0,0,0,1851285,182,1116633,0,734652,0,1116633,734652,1.0,181.0
1,rpr,1851285,0,0,0,0,1851285,186,1644176,0,207109,0,1644176,207109,1.0,775.0
2,ts,1851285,0,0,0,0,1851285,179,1227384,0,623901,0,1227384,623901,1.0,178.0


## Stage 4 — Inspect text vocabularies and extreme numeric values

The source-wide storage profile shows that every row contains a stored value in all three rating fields, but hundreds of thousands of those values are stored as text.

The exact single dash `-` does not occur, despite the provisional governance policy referring to an unavailable-rating dash. This means the actual text vocabulary must be inspected before availability can be classified.

This stage will therefore:

- list every distinct text value and its frequency for `or`, `rpr` and `ts`;
- preserve whitespace and raw representation visibly;
- identify whether missing ratings use another dash form or token;
- inspect the highest and lowest physically numeric values;
- retain source row lineage for suspicious extremes.

No text value will yet be parsed or recoded. In particular, repeated punctuation, unusual symbols and numeric-looking text will remain distinct until their observed behaviour is understood.

In [5]:
# Collect the complete text vocabulary for each rating field.
#
# The previous profile showed that text storage is substantial, so we must
# inspect the actual raw tokens before deciding which values mean unavailable,
# malformed, numeric-looking, or something else.
text_vocabulary_queries = []

for field in RATING_FIELDS:
    # Quote the column identifier because `or` is a SQLite keyword.
    quoted_field = f'"{field}"'

    text_vocabulary_queries.append(
        f"""
        SELECT
            '{field}' AS source_field,

            -- Preserve the original text exactly as stored.
            CAST({quoted_field} AS TEXT) AS raw_value,

            -- Show the text length so invisible whitespace or repeated
            -- punctuation can be distinguished from visually similar tokens.
            LENGTH(CAST({quoted_field} AS TEXT)) AS raw_length,

            -- Count how often each exact raw text value occurs.
            COUNT(*) AS runner_rows

        FROM "{SOURCE_TABLE}"
        WHERE {DATA_ROW_PREDICATE}
          AND typeof({quoted_field}) = 'text'

        GROUP BY
            CAST({quoted_field} AS TEXT),
            LENGTH(CAST({quoted_field} AS TEXT))
        """
    )

# Combine the three field vocabularies into one result while preserving the
# source-field identity of every raw token.
text_vocabulary_sql = "\nUNION ALL\n".join(text_vocabulary_queries)

rating_text_vocabulary = pd.read_sql_query(
    text_vocabulary_sql,
    connection,
)

# Reconcile the vocabulary frequencies back to the text-storage counts from
# Stage 3. This protects against accidentally excluding any text value.
observed_text_counts = (
    rating_text_vocabulary
    .groupby("source_field", as_index=False)["runner_rows"]
    .sum()
    .rename(columns={"runner_rows": "vocabulary_text_rows"})
)

expected_text_counts = rating_storage_profile[
    ["source_field", "text_storage_rows"]
].copy()

text_count_reconciliation = expected_text_counts.merge(
    observed_text_counts,
    on="source_field",
    how="left",
    validate="one_to_one",
)

assert text_count_reconciliation["vocabulary_text_rows"].eq(
    text_count_reconciliation["text_storage_rows"]
).all()

# Sort first by field order, then by descending frequency and raw value so the
# dominant availability tokens appear clearly.
field_order = {field: position for position, field in enumerate(RATING_FIELDS)}

rating_text_vocabulary["field_order"] = (
    rating_text_vocabulary["source_field"].map(field_order)
)

rating_text_vocabulary = (
    rating_text_vocabulary
    .sort_values(
        ["field_order", "runner_rows", "raw_value"],
        ascending=[True, False, True],
    )
    .drop(columns="field_order")
    .reset_index(drop=True)
)

# Inspect numeric extremes separately. Returning source lineage and race
# context allows suspicious values, especially the observed rpr maximum of
# 775, to be located without altering or normalising them.
extreme_value_queries = []

for field in RATING_FIELDS:
    quoted_field = f'"{field}"'

    extreme_value_queries.append(
        f"""
        SELECT *
        FROM (
            SELECT
                '{field}' AS source_field,
                'lowest' AS extreme_type,
                rowid AS source_rowid,
                date,
                course,
                off,
                race_id,
                horse,
                type AS race_type,
                {quoted_field} AS raw_value,
                typeof({quoted_field}) AS storage_class
            FROM "{SOURCE_TABLE}"
            WHERE {DATA_ROW_PREDICATE}
              AND typeof({quoted_field}) IN ('integer', 'real')
            ORDER BY CAST({quoted_field} AS REAL) ASC, rowid ASC
            LIMIT 10
        )

        UNION ALL

        SELECT *
        FROM (
            SELECT
                '{field}' AS source_field,
                'highest' AS extreme_type,
                rowid AS source_rowid,
                date,
                course,
                off,
                race_id,
                horse,
                type AS race_type,
                {quoted_field} AS raw_value,
                typeof({quoted_field}) AS storage_class
            FROM "{SOURCE_TABLE}"
            WHERE {DATA_ROW_PREDICATE}
              AND typeof({quoted_field}) IN ('integer', 'real')
            ORDER BY CAST({quoted_field} AS REAL) DESC, rowid ASC
            LIMIT 10
        )
        """
    )

rating_extreme_values = pd.read_sql_query(
    "\nUNION ALL\n".join(extreme_value_queries),
    connection,
)

# Confirm that each field returned ten low and ten high examples.
extreme_counts = (
    rating_extreme_values
    .groupby(["source_field", "extreme_type"])
    .size()
)

for field in RATING_FIELDS:
    assert extreme_counts.loc[(field, "lowest")] == 10
    assert extreme_counts.loc[(field, "highest")] == 10

print("Complete text vocabulary")
display(rating_text_vocabulary)

print("Physically numeric extreme values with source lineage")
display(rating_extreme_values)

Complete text vocabulary


,source_field,raw_value,raw_length,runner_rows
0,or,–,1,734652
1,rpr,–,1,207109
2,ts,–,1,623901


Physically numeric extreme values with source lineage


,source_field,extreme_type,source_rowid,date,course,off,race_id,horse,race_type,raw_value,storage_class
0,or,lowest,128881,2015-10-18,Bath,3:35,635392,A Definite Diamond (GB),Flat,1,integer
1,or,lowest,308525,2016-12-13,Southwell (AW),2:00,663682,Striking Nigella (GB),Flat,1,integer
2,or,lowest,607263,2018-09-26,Redcar,4:30,710662,No Civil Justice (GB),Flat,1,integer
3,or,lowest,744006,2019-07-10,Yarmouth,2:50,732838,Intimate Moment (GB),Flat,1,integer
4,or,lowest,746549,2019-07-15,Wolverhampton (AW),6:10,734127,Chateau Peapod (GB),Flat,1,integer
5,or,lowest,753479,2019-07-30,Yarmouth,2:15,734758,Paisleys Promise (IRE),Flat,1,integer
6,or,lowest,753486,2019-07-30,Yarmouth,2:15,734758,Ikigai (GB),Flat,1,integer
7,or,lowest,754343,2019-08-01,Nottingham,2:50,734824,Ximena (GB),Flat,1,integer
8,or,lowest,770086,2019-09-02,Windsor,7:00,736917,Aspiring Diva (GB),Flat,1,integer
9,or,lowest,780981,2019-09-24,Lingfield (AW),2:20,739000,Alizes (FR),Flat,1,integer


## Stage 5 — Investigate the isolated `rpr = 775` anomaly

The complete raw vocabulary establishes that unavailable ratings use the exact Unicode en dash `–`.

The physically numeric ranges are:

- `or`: 1 to 181;
- `rpr`: 1 to 775;
- `ts`: 1 to 178.

However, the next-highest observed `rpr` after 775 is 184. The value 775 is therefore an isolated extreme rather than part of a continuous upper range.

This stage will inspect:

- the complete provisional race containing the value;
- all source fields for the affected runner;
- adjacent physical source rows;
- other source appearances of the same horse;
- whether 775 resembles another value present in the row or race.

The raw value will remain unchanged. This stage is diagnostic only and will not yet classify the value as a confirmed error or assign a replacement.

In [6]:
# Preserve the exact lineage identified in the Stage 4 extreme-value output.
SUSPICIOUS_RPR_SOURCE_ROWID = 1_619_851

# Load the complete affected source row so the suspicious value can be checked
# against every other stored field without assuming the cause of the anomaly.
suspicious_rpr_row = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        *
    FROM "{SOURCE_TABLE}"
    WHERE rowid = ?
      AND {DATA_ROW_PREDICATE}
    """,
    connection,
    params=[SUSPICIOUS_RPR_SOURCE_ROWID],
)

# The lineage lookup must return exactly one governed source row.
assert len(suspicious_rpr_row) == 1
assert int(suspicious_rpr_row.loc[0, "rpr"]) == 775

# Extract the provisional race identity and horse label directly from the
# source row rather than retyping them into later queries.
suspicious_date = suspicious_rpr_row.loc[0, "date"]
suspicious_course = suspicious_rpr_row.loc[0, "course"]
suspicious_off = suspicious_rpr_row.loc[0, "off"]
suspicious_horse = suspicious_rpr_row.loc[0, "horse"]

# Load every runner in the same provisional race. This shows whether the value
# belongs to a broader race-level formatting problem or is isolated to one row.
suspicious_rpr_race = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_id,
        race_name,
        type AS race_type,
        pos,
        horse,
        age,
        sex,
        "or" AS raw_or,
        rpr AS raw_rpr,
        ts AS raw_ts,
        sp,
        comment
    FROM "{SOURCE_TABLE}"
    WHERE {DATA_ROW_PREDICATE}
      AND date = ?
      AND course = ?
      AND off = ?
    ORDER BY rowid
    """,
    connection,
    params=[
        suspicious_date,
        suspicious_course,
        suspicious_off,
    ],
)

# Confirm that the suspicious row remains present once the complete race is
# reconstructed from date, course and off time.
assert SUSPICIOUS_RPR_SOURCE_ROWID in set(
    suspicious_rpr_race["source_rowid"]
)

# Inspect a narrow physical-row window around the anomaly. Adjacent rowids can
# expose shifted, concatenated or repeated extraction values even when those
# rows belong to different races.
suspicious_rpr_neighbours = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_id,
        horse,
        pos,
        "or" AS raw_or,
        rpr AS raw_rpr,
        ts AS raw_ts
    FROM "{SOURCE_TABLE}"
    WHERE rowid BETWEEN ? AND ?
      AND {DATA_ROW_PREDICATE}
    ORDER BY rowid
    """,
    connection,
    params=[
        SUSPICIOUS_RPR_SOURCE_ROWID - 5,
        SUSPICIOUS_RPR_SOURCE_ROWID + 5,
    ],
)

# Load every other appearance of the same horse so the suspicious value can be
# compared with its own source history. This does not establish the correct
# rating, but it can show whether 775 recurs or is unique.
suspicious_horse_history = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_id,
        race_name,
        type AS race_type,
        pos,
        horse,
        "or" AS raw_or,
        rpr AS raw_rpr,
        ts AS raw_ts
    FROM "{SOURCE_TABLE}"
    WHERE {DATA_ROW_PREDICATE}
      AND horse = ?
    ORDER BY date, rowid
    """,
    connection,
    params=[suspicious_horse],
)

# Protect the diagnostic scope: the source-wide profile showed only one value
# above the otherwise observed maximum of 184.
rpr_above_184_count = pd.read_sql_query(
    f"""
    SELECT COUNT(*) AS runner_rows
    FROM "{SOURCE_TABLE}"
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(rpr) IN ('integer', 'real')
      AND CAST(rpr AS REAL) > 184
    """,
    connection,
)

assert int(rpr_above_184_count.loc[0, "runner_rows"]) == 1

print("Complete suspicious source row")
display(suspicious_rpr_row.T)

print("Complete provisional race containing rpr = 775")
display(suspicious_rpr_race)

print("Adjacent physical source rows")
display(suspicious_rpr_neighbours)

print(f"Other source appearances of {suspicious_horse}")
display(suspicious_horse_history)

Complete suspicious source row


,0
source_rowid,1619851
date,2025-01-03
course,Deauville (FR)
race_id,885653
off,4:27
race_name,Prix de la Seulette (Handicap) (4yo) (All-Weat...
type,Flat
class,
pattern,
rating_band,


Complete provisional race containing rpr = 775


,source_rowid,date,course,off,race_id,race_name,race_type,pos,horse,age,sex,raw_or,raw_rpr,raw_ts,sp,comment
0,1619850,2025-01-03,Deauville (FR),4:27,885653,Prix de la Seulette (Handicap) (4yo) (All-Weat...,Flat,2,Intuition (FR),4,F,–,68,46,56/10,
1,1619851,2025-01-03,Deauville (FR),4:27,885653,Prix de la Seulette (Handicap) (4yo) (All-Weat...,Flat,1,Si Capo Si (FR),4,G,–,775,50,37/10F,
2,1619869,2025-01-03,Deauville (FR),4:27,885653,Prix de la Seulette (Handicap) (4yo) (All-Weat...,Flat,3,Baileys Bachelor (FR),4,G,–,68,46,36/1,
3,1619871,2025-01-03,Deauville (FR),4:27,885653,Prix de la Seulette (Handicap) (4yo) (All-Weat...,Flat,4,Costarmoricain (IRE),4,G,–,69,47,11/1,
4,1619872,2025-01-03,Deauville (FR),4:27,885653,Prix de la Seulette (Handicap) (4yo) (All-Weat...,Flat,6,Baileys Vitesse (FR),4,G,–,65,43,6/1,
5,1619885,2025-01-03,Deauville (FR),4:27,885653,Prix de la Seulette (Handicap) (4yo) (All-Weat...,Flat,13,Bakhilova (FR),4,F,–,30,9,16/1,
6,1619886,2025-01-03,Deauville (FR),4:27,885653,Prix de la Seulette (Handicap) (4yo) (All-Weat...,Flat,12,Fortunate Son (FR),4,G,–,55,34,10/1,
7,1619887,2025-01-03,Deauville (FR),4:27,885653,Prix de la Seulette (Handicap) (4yo) (All-Weat...,Flat,11,Rum Tum Tugger (FR),4,G,–,39,18,70/1,
8,1619888,2025-01-03,Deauville (FR),4:27,885653,Prix de la Seulette (Handicap) (4yo) (All-Weat...,Flat,10,Picanes (FR),4,F,–,54,33,21/1,
9,1619889,2025-01-03,Deauville (FR),4:27,885653,Prix de la Seulette (Handicap) (4yo) (All-Weat...,Flat,9,Best Looking (FR),4,F,–,65,43,6/1,


Adjacent physical source rows


,source_rowid,date,course,off,race_id,horse,pos,raw_or,raw_rpr,raw_ts
0,1619846,2025-01-03,Meydan (UAE),2:05,885541,Cupids Dream (GB),7,85,73,57
1,1619847,2025-01-03,Deauville (FR),12:57,885619,Bahia Warana (FR),5,–,48,–
2,1619848,2025-01-03,Meydan (UAE),1:30,885540,Home Brew (USA),9,101,99,78
3,1619849,2025-01-03,Deauville (FR),12:57,885619,Lexovienne (FR),7,–,47,–
4,1619850,2025-01-03,Deauville (FR),4:27,885653,Intuition (FR),2,–,68,46
5,1619851,2025-01-03,Deauville (FR),4:27,885653,Si Capo Si (FR),1,–,775,50
6,1619852,2025-01-03,Deauville (FR),3:52,885637,Royal Enclosure (GB),14,–,29,18
7,1619853,2025-01-03,Deauville (FR),3:52,885637,Bradiancourt (FR),13,–,34,23
8,1619854,2025-01-03,Deauville (FR),3:52,885637,Posafolie (FR),12,–,59,48
9,1619855,2025-01-03,Deauville (FR),3:52,885637,Hey Vince (FR),11,–,65,53


Other source appearances of Si Capo Si (FR)


,source_rowid,date,course,off,race_id,race_name,race_type,pos,horse,raw_or,raw_rpr,raw_ts
0,1544482,2024-07-30,Deauville (FR),3:53,873680,Prix de la Galopiniere (Claimer) (3yo) (All-We...,Flat,14,Si Capo Si (FR),–,–,16
1,1552665,2024-08-17,Deauville (FR),1:33,875180,Prix dIsigny (Claimer) (3yo) (All-Weather Trac...,Flat,3,Si Capo Si (FR),–,73,49
2,1561244,2024-09-06,Saint-Cloud (FR),4:10,876422,Prix de Colombes (Claimer) (3yo) (Turf),Flat,9,Si Capo Si (FR),–,–,–
3,1581961,2024-10-15,Chantilly (FR),2:05,879626,Prix du Chene Sylvie (Handicap) (3yo) (All-Wea...,Flat,8,Si Capo Si (FR),–,46,26
4,1587240,2024-10-24,Deauville (FR),2:22,879982,Prix de lEcluse Francois I ER (Handicap) (3yo)...,Flat,2,Si Capo Si (FR),–,–,29
5,1603223,2024-11-26,Deauville (FR),6:12,882955,Prix du Bois dEnfer (Handicap) (3yo) (All-Weat...,Flat,2,Si Capo Si (FR),–,56,51
6,1610366,2024-12-13,Deauville (FR),3:55,884120,Prix des Mas (Handicap) (3yo) (All-Weather Tra...,Flat,10,Si Capo Si (FR),–,56,20
7,1618342,2024-12-30,Chantilly (FR),2:57,885142,Prix de la Porte Nointel (Handicap) (3yo) (All...,Flat,3,Si Capo Si (FR),–,62,40
8,1619851,2025-01-03,Deauville (FR),4:27,885653,Prix de la Seulette (Handicap) (4yo) (All-Weat...,Flat,1,Si Capo Si (FR),–,775,50
9,1716350,2025-08-07,Deauville (FR),5:05,901270,Prix du Mesnil-Eudes (Handicap) (4yo+) (All-We...,Flat,14,Si Capo Si (FR),–,28,10


## Stage 6 — Govern the isolated invalid RPR value

The source contains one isolated `rpr` value of `775`:

- source rowid: `1619851`;
- date: 3 January 2025;
- course: Deauville (FR);
- off time: 4:27;
- horse: Si Capo Si (FR).

This value is not treated as a valid RPR.

The source-internal evidence is decisive:

- it is the only `rpr` above 184 in 1,851,285 governed runner rows;
- the next-highest observed `rpr` is 184;
- the remaining runners in the same race have RPR values between 30 and 71;
- the value is inconsistent with the observed RPR scale and distribution.

External representations of the exact published result also contradict `775`, although they do not currently provide sufficiently consistent evidence to establish whether the intended value was `75` or unavailable.

The governed decision is therefore:

- preserve the immutable raw value `775`;
- do not expose `775` as a usable parsed RPR;
- set the analytical RPR to null for this exact source row;
- classify the row as `invalid_source_value`;
- leave the replacement value unresolved;
- apply the rule only through exact physical source lineage.

The downstream representation is:

- `raw_rpr = 775`
- `parsed_rpr = null`
- `availability_status = invalid_source_value`
- `replacement_status = unresolved`
- `source_rowid = 1619851`

This is not a general rule that all unusually high ratings should be removed. It is an exact exclusion rule for one uniquely identified source defect.

All other physically numeric values remain rating candidates until later jurisdiction, race-type, temporal and cross-field analysis establishes their analytical limitations.

In [7]:
# Define the exact raw token used by the source when a rating is unavailable.
#
# The source uses a Unicode en dash rather than an ASCII hyphen.
UNAVAILABLE_RATING_TOKEN = "–"

# Preserve the exact physical lineage of the one confirmed invalid RPR value.
#
# The rule is deliberately tied to source rowid rather than applied generally
# to every value equal to 775.
INVALID_RPR_SOURCE_ROWID = 1_619_851
INVALID_RPR_RAW_VALUE = 775

# Build one complete source-wide availability query for each rating field.
rating_availability_queries = []

for field in RATING_FIELDS:
    # Quote the source identifier because `or` is a reserved SQLite keyword.
    quoted_field = f'"{field}"'

    # Only rpr currently has an exact lineage-backed invalid source value.
    #
    # For or and ts, the condition is always false.
    if field == "rpr":
        invalid_value_condition = (
            f"rowid = {INVALID_RPR_SOURCE_ROWID} "
            f"AND {quoted_field} = {INVALID_RPR_RAW_VALUE}"
        )
    else:
        invalid_value_condition = "0"

    rating_availability_queries.append(
        f"""
        SELECT
            '{field}' AS source_field,

            -- Partition every raw source value into one structural state.
            CASE
                -- Apply the exact invalid-value rule before the general
                -- physically numeric rule so rpr=775 cannot enter analysis.
                WHEN {invalid_value_condition}
                THEN 'invalid_source_value'

                -- Preserve the exact source token representing an unavailable
                -- rating.
                WHEN typeof({quoted_field}) = 'text'
                 AND CAST({quoted_field} AS TEXT) = ?
                THEN 'unavailable_en_dash'

                -- All remaining integer or real values are numeric candidates.
                -- This does not yet establish cross-field or cross-jurisdiction
                -- comparability.
                WHEN typeof({quoted_field}) IN ('integer', 'real')
                THEN 'numeric_candidate'

                -- Retain any unexpected future representation explicitly.
                ELSE 'unresolved_other'
            END AS availability_status,

            COUNT(*) AS runner_rows,

            -- Count distinct raw values contributing to each state without
            -- normalising or replacing the source representation.
            COUNT(DISTINCT {quoted_field}) AS distinct_raw_values,

            -- Calculate candidate ranges only from values allowed into the
            -- usable numeric population.
            MIN(
                CASE
                    WHEN NOT ({invalid_value_condition})
                     AND typeof({quoted_field}) IN ('integer', 'real')
                    THEN CAST({quoted_field} AS INTEGER)
                END
            ) AS min_numeric_candidate,

            MAX(
                CASE
                    WHEN NOT ({invalid_value_condition})
                     AND typeof({quoted_field}) IN ('integer', 'real')
                    THEN CAST({quoted_field} AS INTEGER)
                END
            ) AS max_numeric_candidate

        FROM "{SOURCE_TABLE}"
        WHERE {DATA_ROW_PREDICATE}

        GROUP BY
            CASE
                WHEN {invalid_value_condition}
                THEN 'invalid_source_value'
                WHEN typeof({quoted_field}) = 'text'
                 AND CAST({quoted_field} AS TEXT) = ?
                THEN 'unavailable_en_dash'
                WHEN typeof({quoted_field}) IN ('integer', 'real')
                THEN 'numeric_candidate'
                ELSE 'unresolved_other'
            END
        """
    )

# Each field query contains two placeholders for the en-dash token:
# one in the selected CASE expression and one in the matching GROUP BY.
availability_parameters = [
    UNAVAILABLE_RATING_TOKEN,
    UNAVAILABLE_RATING_TOKEN,
] * len(RATING_FIELDS)

# Combine the three field-specific profiles into one result.
rating_availability_sql = "\nUNION ALL\n".join(
    rating_availability_queries
)

rating_availability_profile = pd.read_sql_query(
    rating_availability_sql,
    connection,
    params=availability_parameters,
)

# Apply a stable display order rather than relying on SQLite grouping order.
field_order = {
    field: position
    for position, field in enumerate(RATING_FIELDS)
}

status_order = {
    "numeric_candidate": 0,
    "unavailable_en_dash": 1,
    "invalid_source_value": 2,
    "unresolved_other": 3,
}

rating_availability_profile["field_order"] = (
    rating_availability_profile["source_field"].map(field_order)
)

rating_availability_profile["status_order"] = (
    rating_availability_profile["availability_status"].map(status_order)
)

rating_availability_profile = (
    rating_availability_profile
    .sort_values(["field_order", "status_order"])
    .drop(columns=["field_order", "status_order"])
    .reset_index(drop=True)
)

# Confirm that each field still partitions exactly to the complete governed
# runner population.
population_reconciliation = (
    rating_availability_profile
    .groupby("source_field", as_index=False)["runner_rows"]
    .sum()
)

assert population_reconciliation["source_field"].tolist() == RATING_FIELDS
assert population_reconciliation["runner_rows"].eq(
    EXPECTED_RUNNER_ROWS
).all()

# No observed source value should remain outside the governed states.
unresolved_rows = rating_availability_profile.loc[
    rating_availability_profile["availability_status"]
    == "unresolved_other",
    "runner_rows",
].sum()

assert int(unresolved_rows) == 0

# Protect the exact invalid-value cardinality.
invalid_rows = rating_availability_profile.loc[
    rating_availability_profile["availability_status"]
    == "invalid_source_value",
    "runner_rows",
].sum()

assert int(invalid_rows) == 1

# Confirm the candidate numeric ranges after excluding the invalid RPR row.
numeric_ranges = (
    rating_availability_profile.loc[
        rating_availability_profile["availability_status"]
        == "numeric_candidate",
        [
            "source_field",
            "min_numeric_candidate",
            "max_numeric_candidate",
        ],
    ]
    .set_index("source_field")
)

assert int(numeric_ranges.loc["or", "min_numeric_candidate"]) == 1
assert int(numeric_ranges.loc["or", "max_numeric_candidate"]) == 181

assert int(numeric_ranges.loc["rpr", "min_numeric_candidate"]) == 1
assert int(numeric_ranges.loc["rpr", "max_numeric_candidate"]) == 184

assert int(numeric_ranges.loc["ts", "min_numeric_candidate"]) == 1
assert int(numeric_ranges.loc["ts", "max_numeric_candidate"]) == 178

# Display the complete governed partition before moving to temporal,
# jurisdiction, race-type or cross-field availability.
display(rating_availability_profile)

,source_field,availability_status,runner_rows,distinct_raw_values,min_numeric_candidate,max_numeric_candidate
0,or,numeric_candidate,1116633,181,1.0,181.0
1,or,unavailable_en_dash,734652,1,NaN,NaN
2,rpr,numeric_candidate,1644175,184,1.0,184.0
3,rpr,unavailable_en_dash,207109,1,NaN,NaN
4,rpr,invalid_source_value,1,1,NaN,NaN
5,ts,numeric_candidate,1227384,178,1.0,178.0
6,ts,unavailable_en_dash,623901,1,NaN,NaN


## Stage 7 — Profile cross-field rating availability

The three rating fields have different source-wide coverage:

- `rpr` is available most often;
- `ts` is unavailable more frequently;
- `or` is unavailable most frequently;
- one exact `rpr` row is excluded as an invalid source value.

Field-level counts do not show whether the ratings are missing independently or in recurring combinations.

This stage will classify every governed runner row by whether each field has a usable numeric candidate.

The resulting patterns will establish how often:

- all three ratings are available;
- only two ratings are available;
- only one rating is available;
- no usable rating is available;
- the invalid `rpr` row fits into the wider availability pattern.

For this stage:

- the Unicode en dash remains unavailable;
- the exact `rpr = 775` row remains invalid and non-numeric;
- availability does not imply that the three scales are comparable;
- co-occurrence does not prove that the ratings were produced at the same analytical time.

## Stage 7 — Profile cross-field rating availability

The three rating fields have different source-wide coverage:

- `rpr` is available most often;
- `ts` is unavailable more frequently;
- `or` is unavailable most frequently;
- one exact `rpr` row is excluded as an invalid source value.

Field-level counts do not show whether the ratings are missing independently or in recurring combinations.

This stage will classify every governed runner row by whether each field has a usable numeric candidate.

The resulting patterns will establish how often:

- all three ratings are available;
- only two ratings are available;
- only one rating is available;
- no usable rating is available;
- the invalid `rpr` row fits into the wider availability pattern.

For this stage:

- the Unicode en dash remains unavailable;
- the exact `rpr = 775` row remains invalid and non-numeric;
- availability does not imply that the three scales are comparable;
- co-occurrence does not prove that the ratings were produced at the same analytical time.

In [8]:
# Build one runner-level availability pattern across or, rpr and ts.
#
# A field is marked available only when it contains a physically numeric value
# that is allowed into the candidate population. The exact invalid rpr row is
# therefore treated as unavailable for analytical purposes.
rating_availability_patterns = pd.read_sql_query(
    f"""
    WITH runner_rating_states AS (
        SELECT
            rowid AS source_rowid,

            -- OR has no known invalid numeric exceptions, so any integer or
            -- real storage value is currently a numeric candidate.
            CASE
                WHEN typeof("or") IN ('integer', 'real')
                THEN 1
                ELSE 0
            END AS or_available,

            -- Exclude the exact lineage-backed rpr=775 defect before marking
            -- physically numeric RPR values as available.
            CASE
                WHEN rowid = ?
                 AND rpr = ?
                THEN 0
                WHEN typeof(rpr) IN ('integer', 'real')
                THEN 1
                ELSE 0
            END AS rpr_available,

            -- TS has no known invalid numeric exceptions at this stage.
            CASE
                WHEN typeof(ts) IN ('integer', 'real')
                THEN 1
                ELSE 0
            END AS ts_available,

            -- Preserve the anomaly as a separate flag so its combination can
            -- be inspected rather than disappearing inside the unavailable
            -- population.
            CASE
                WHEN rowid = ?
                 AND rpr = ?
                THEN 1
                ELSE 0
            END AS invalid_rpr_flag

        FROM "{SOURCE_TABLE}"
        WHERE {DATA_ROW_PREDICATE}
    )

    SELECT
        or_available,
        rpr_available,
        ts_available,

        -- Count how many of the three ratings are analytically available on
        -- each runner row.
        (
            or_available
            + rpr_available
            + ts_available
        ) AS available_rating_count,

        -- Create a readable pattern label without collapsing field identity.
        CASE
            WHEN or_available = 1
             AND rpr_available = 1
             AND ts_available = 1
            THEN 'or+rpr+ts'

            WHEN or_available = 1
             AND rpr_available = 1
             AND ts_available = 0
            THEN 'or+rpr'

            WHEN or_available = 1
             AND rpr_available = 0
             AND ts_available = 1
            THEN 'or+ts'

            WHEN or_available = 0
             AND rpr_available = 1
             AND ts_available = 1
            THEN 'rpr+ts'

            WHEN or_available = 1
             AND rpr_available = 0
             AND ts_available = 0
            THEN 'or_only'

            WHEN or_available = 0
             AND rpr_available = 1
             AND ts_available = 0
            THEN 'rpr_only'

            WHEN or_available = 0
             AND rpr_available = 0
             AND ts_available = 1
            THEN 'ts_only'

            ELSE 'none'
        END AS availability_pattern,

        COUNT(*) AS runner_rows,

        -- Confirm whether the exact invalid RPR row belongs to this pattern.
        SUM(invalid_rpr_flag) AS invalid_rpr_rows

    FROM runner_rating_states

    GROUP BY
        or_available,
        rpr_available,
        ts_available

    ORDER BY
        available_rating_count DESC,
        availability_pattern
    """,
    connection,
    params=[
        INVALID_RPR_SOURCE_ROWID,
        INVALID_RPR_RAW_VALUE,
        INVALID_RPR_SOURCE_ROWID,
        INVALID_RPR_RAW_VALUE,
    ],
)

# Every governed row must appear in exactly one cross-field pattern.
assert int(
    rating_availability_patterns["runner_rows"].sum()
) == EXPECTED_RUNNER_ROWS

# The eight possible binary combinations should form a complete partition.
assert len(rating_availability_patterns) <= 8

# Each pattern label must be unique because it represents one exact
# availability combination.
assert rating_availability_patterns[
    "availability_pattern"
].is_unique

# The exact invalid RPR row must remain visible exactly once.
assert int(
    rating_availability_patterns["invalid_rpr_rows"].sum()
) == 1

# Add source-wide percentages for easier interpretation while retaining the
# exact runner counts.
rating_availability_patterns["runner_percentage"] = (
    rating_availability_patterns["runner_rows"]
    / EXPECTED_RUNNER_ROWS
    * 100
).round(4)

display(rating_availability_patterns)

,or_available,rpr_available,ts_available,available_rating_count,availability_pattern,runner_rows,invalid_rpr_rows,runner_percentage
0,1,1,1,3,or+rpr+ts,847923,0,45.8019
1,1,1,0,2,or+rpr,180205,0,9.7340
2,1,0,1,2,or+ts,5146,0,0.2780
3,0,1,1,2,rpr+ts,365667,0,19.7521
4,1,0,0,1,or_only,83359,0,4.5028
5,0,1,0,1,rpr_only,250380,0,13.5247
6,0,0,1,1,ts_only,8648,1,0.4671
7,0,0,0,0,none,109957,0,5.9395


## Stage 7 conclusion — Availability is structured, not universal

The cross-field profile was used only to test whether rating availability behaves as one shared all-or-nothing state.

It does not.

Only 847,923 runner rows, or 45.80% of the governed source, contain usable numeric candidates in all three fields.

`rpr` is available much more widely than either `or` or `ts`, while substantial populations contain only one or two of the three ratings.

The practical consequence is:

- missingness must remain field-specific;
- analyses requiring all three ratings would use a selected minority of the source;
- one generic `rating_available` flag would lose important information;
- no further combination taxonomy is required at this stage.

The next investigation returns to the bounded semantic question: how availability varies by period, jurisdiction and race type, and whether those patterns explain the different field coverage.

## Stage 7 conclusion — Ratings must remain independent fields

The cross-field table confirms that rating availability is not one shared all-or-nothing condition.

This is expected because the fields have different producers and purposes:

- `or` is an official handicap mark assigned by the relevant racing authority;
- `rpr` is a proprietary Racing Post assessment of an individual performance;
- `ts` is a proprietary Racing Post speed figure.

A horse may therefore have any one of these fields without necessarily having the others.

The database must:

- preserve separate nullable columns for `or`, `rpr` and `ts`;
- preserve the source en dash as an unavailable state;
- exclude the exact invalid `rpr = 775` value from the analytical RPR;
- avoid requiring all three fields for a runner record to be usable;
- avoid creating one generic `rating_available` field.

No further cross-field combination analysis is required.

## Stage 8 — Establish field meaning and timing

The three rating fields represent different kinds of information and must not be treated as interchangeable.

### `or`

`or` is the official handicap rating applicable to the horse for that race.

For British racing, official ratings are produced by the British Horseracing Authority and are used to allocate weights in handicap races.

This is therefore a pre-race or current-state field: it records the official mark the horse was running from, not a retrospective assessment of how it performed in that race.

### `rpr`

`rpr` is the Racing Post Rating awarded to the horse for its performance in that completed race.

Racing Post describes RPR as a measure of performance and explains that races are usually rated after the event, normally the following morning. The result rating shows how much better or worse the horse performed relative to the other runners.

This is therefore a retrospective performance rating.

RPRs are not necessarily permanently fixed. Racing Post states that past result ratings may later be raised or lowered as subsequent form changes the overall assessment.

### `ts`

`ts` is Racing Post's retrospective speed figure for the horse's performance in that completed race.

Speed ratings estimate how fast a horse ran on a particular day. They are derived from the completed race time, standard times, ground conditions, race distance and beaten distances.

This is therefore a retrospective performance figure rather than a pre-race prediction.

### Governed interpretation

The database should interpret the fields as:

- `or`: official pre-race handicap mark;
- `rpr`: retrospective and potentially revisable Racing Post performance rating;
- `ts`: retrospective Racing Post speed figure.

The fields must remain separate because they have different producers, purposes and timing.

## Final conclusion

Notebook 18 investigated the meaning, storage, availability and analytical treatment of the source fields `or`, `rpr` and `ts`.

### Executive conclusion

The three fields are valid rating fields, but they do not represent the same kind of information and must remain separate in the future database.

- `or` is the official handicap mark applicable to the horse for that race.
- `rpr` is Racing Post's retrospective assessment of the horse's performance in that completed race.
- `ts` is Racing Post's retrospective speed figure for that completed performance.

The Racing Post handbook confirms that RPRs are normally compiled after the race and that speed ratings estimate how fast a horse ran on a particular day. It also states that past RPRs can later be revised as subsequent form changes the overall assessment. 

### Source representation

Across the governed population of 1,851,285 runner rows:

- all three fields are physically populated;
- available ratings are stored as integers;
- unavailable ratings are stored as the Unicode en dash `–`;
- no blanks, nulls, ASCII hyphens or other text tokens were observed.

The unavailable token must therefore be interpreted as:

- raw value: `–`;
- parsed analytical value: `NULL`;
- status: `unavailable`.

It must never be converted to zero.

### Numeric candidate ranges

After applying the governed exception described below, the observed numeric candidate ranges are:

- `or`: 1 to 181;
- `rpr`: 1 to 184;
- `ts`: 1 to 178.

These are observed source ranges, not universal validity rules for all future data.

### Isolated invalid RPR value

One runner row stores `rpr = 775`:

- source rowid: `1619851`;
- date: 3 January 2025;
- course: Deauville (FR);
- off time: 4:27;
- horse: Si Capo Si (FR).

This is the only RPR above 184 in the complete governed source population. The remaining runners in the same race and the horse's surrounding performances are on the normal rating scale.

The value is therefore treated as an invalid source value rather than a legitimate rating.

The governed treatment is:

- preserve `raw_rpr = 775`;
- set the parsed analytical RPR to `NULL`;
- record status `invalid_source_value`;
- retain exact source-row lineage;
- leave the intended replacement unresolved.

The value must not be corrected to 75 or any other figure without reliable exact-race evidence.

### Availability

Rating availability is field-specific rather than all-or-nothing.

Only 847,923 runner rows, or 45.80% of the governed population, contain numeric candidates in all three fields.

This is expected because the fields have different producers, purposes and timing. No generic `rating_available` field should replace the three independent nullable fields.

### Database consequence

The future database must preserve:

- the three raw source values;
- separate nullable parsed integer columns for `or`, `rpr` and `ts`;
- a separate status for each field;
- exact source lineage;
- the specific invalid-RPR exception for source rowid `1619851`.

The database must not:

- overwrite raw values;
- convert unavailable ratings to zero;
- treat the three rating fields as interchangeable;
- require all three ratings for a runner record to be analytically usable;
- expose `rpr = 775` as a numeric analytical value.

### Confidence and limitations

Confidence is high for:

- the physical source representation;
- the source-wide counts and ranges;
- the meaning and timing of the three fields;
- the conclusion that `rpr = 775` is analytically invalid.

The intended replacement for `rpr = 775` remains unresolved.

This notebook did not investigate:

- predictive value;
- profitability;
- jurisdiction-specific coverage;
- race-type coverage;
- revisions to historical RPRs over time;
- whether RPR or TS adds information beyond the other fields.

Those are separate analytical questions and are not required for the database-governance decision.

### Practical implication

The ratings can be retained for later analysis, provided their different meanings, timing and unavailable states are preserved correctly.

Notebook 18 has answered the bounded database question. Further ratings analysis should be conducted only after the governed transformation, tests, validator and integration documentation have been implemented.